# Virchow Training on Google Colab (Resume-Safe)

This notebook runs `scripts/train_virchow_preprocessed_colab.py` and stores checkpoints in Google Drive so training can resume after Colab disconnects.

In [1]:
!fusermount -u /content/drive || true
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

fusermount: failed to unmount /content/drive: No such file or directory
Mounted at /content/drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!nvidia-smi

Thu Apr 23 18:38:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
%cd /content
!git clone https://github.com/LeenRayess/GP_ECG.git GP_ECG
%cd /content/GP_ECG

/content
Cloning into 'GP_ECG'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 168 (delta 15), reused 20 (delta 7), pack-reused 134 (from 1)
Receiving objects: 100% (168/168), 43.77 MiB | 21.69 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/GP_ECG


If you already uploaded your repo into Drive instead of cloning, skip previous cell and set `REPO_DIR` below to that path.

In [5]:
REPO_DIR = '/content/GP_ECG'
PREPROCESSED_DIR = '/content/drive/MyDrive/GP_ECG_DATA/preprocessed_macenko_benchmark_style'
RUN_DIR = '/content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01'
HF_TOKEN = ''  # paste from https://huggingface.co/settings/tokens if needed

import os
os.makedirs(RUN_DIR, exist_ok=True)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

In [6]:
import os
import shutil

# Copy preprocessed .h5 from Google Drive → Colab local SSD (faster reads). RUN_DIR stays on Drive.
LOCAL_PREPROCESSED_DIR = "/content/local_preprocessed_h5"

_src = PREPROCESSED_DIR
if _src.startswith("/content/drive/"):
    if not os.path.isdir(_src):
        raise FileNotFoundError("Drive path not found — mount Drive and check PREPROCESSED_DIR:\n  " + _src)
    if not os.path.exists(LOCAL_PREPROCESSED_DIR):
        print("Copying preprocessed data Drive → local SSD (may take several minutes)...")
        print("  from:", _src)
        shutil.copytree(_src, LOCAL_PREPROCESSED_DIR)
        print("  done:", LOCAL_PREPROCESSED_DIR)
    else:
        print("Using existing local copy:", LOCAL_PREPROCESSED_DIR)
    PREPROCESSED_DIR = LOCAL_PREPROCESSED_DIR
else:
    print("PREPROCESSED_DIR is not under /content/drive/ — using as-is:", _src)

Copying preprocessed data Drive → local SSD (may take several minutes)...
  from: /content/drive/MyDrive/GP_ECG_DATA/preprocessed_macenko_benchmark_style
  done: /content/local_preprocessed_h5


In [7]:
# import os
# print(os.path.isdir("/content/GP_ECG"))

In [8]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 108.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 122.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 106.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 65.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 123.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
import torch
print(torch.__version__)
print("cuda available:", torch.cuda.is_available())

2.5.1+cu121
cuda available: True


In [10]:
%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install timm h5py tqdm huggingface_hub scikit-learn

/content/GP_ECG
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://download.pytorch.org/whl/cu121


Optional if model access is gated: run login once and paste your Hugging Face token.

In [11]:
# !huggingface-cli login

In [12]:
%cd {REPO_DIR}
!python scripts/train_virchow_preprocessed_colab.py \
  --preprocessed-dir "{PREPROCESSED_DIR}" \
  --out-dir "{RUN_DIR}" \
  --epochs 10 \
  --batch-size 256 \
  --num-workers 0 \
  --resume \
  --save-every-epoch-copy

/content/GP_ECG
Device: cuda
GPU: NVIDIA L4
Loading Virchow2 backbone (hf-hub:paige-ai/Virchow2) ...
config.json: 100% 742/742 [00:00<00:00, 2.67MB/s]
model.safetensors: 100% 2.53G/2.53G [00:07<00:00, 339MB/s]
/content/GP_ECG/scripts/train_virchow_preprocessed_colab.py:437: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any u

In [13]:
import json, os
print('Run dir:', RUN_DIR)
artifacts = [
    'checkpoint_last.pt', 'model_best.pt', 'metrics_history.json', 'metrics_final.json',
    'metrics_final_detailed.json', 'temperature_fit.json', 'run_config.json', 'run_manifest.json',
    'run_progress.json', 'val_predictions.npz',
]
for name in artifacts:
    p = os.path.join(RUN_DIR, name)
    print(('OK ' if os.path.exists(p) else 'MISSING '), p)

hist = os.path.join(RUN_DIR, 'metrics_history.json')
if os.path.exists(hist):
    with open(hist, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    if rows:
        print('Last epoch record:', rows[-1])

det = os.path.join(RUN_DIR, 'metrics_final_detailed.json')
if os.path.exists(det):
    with open(det, 'r', encoding='utf-8') as f:
        d = json.load(f)
    print('metrics_final_detailed keys:', list(d.keys()))

Run dir: /content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01
OK  /content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01/checkpoint_last.pt
OK  /content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01/model_best.pt
OK  /content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01/metrics_history.json
OK  /content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01/metrics_final.json
OK  /content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01/metrics_final_detailed.json
OK  /content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01/temperature_fit.json
OK  /content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01/run_config.json
OK  /content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01/run_manifest.json
OK  /content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01/run_progress.json
OK  /content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01/val_predictions.npz
Last epoch record: {'epoch': 10, 'train_loss': 0.1042